# Lab 2：語音進語音出（30 min）

## 目標
- 建立完整的 **STT → LLM → TTS** 語音 Pipeline
- 體驗 OpenAI Whisper（語音轉文字）和 TTS（文字轉語音）API
- 用 LangGraph 將三個步驟串成自動化流程

## 架構
```
🎤 音訊 → [STT Node] → 文字 → [LLM Node] → 回答 → [TTS Node] → 🔊 音訊
```

In [ ]:
%%capture
!pip install langgraph langchain-openai langchain-core openai

In [ ]:
import os
import base64
from pathlib import Path
from typing import TypedDict
from openai import OpenAI
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from IPython.display import Audio, display

# --- API Key 設定 ---
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()
llm = ChatOpenAI(model="gpt-5.4-mini-2026-03-17", temperature=0)

## Step 1: STT — 語音轉文字

先用 TTS 生成一段測試音訊，再用 Whisper 轉回文字。

In [ ]:
# 先生成一段測試音訊（模擬使用者語音輸入）
test_text = "請問 HP EliteBook 855 的電池續航力大概有多久？"

response = client.audio.speech.create(
    model="tts-1",
    voice="nova",
    input=test_text,
)

# 儲存測試音訊
test_audio_path = "test_input.mp3"
response.stream_to_file(test_audio_path)
print(f"測試音訊已生成：{test_audio_path}")
display(Audio(test_audio_path, autoplay=False))

# 用 Whisper 轉錄
with open(test_audio_path, "rb") as audio_file:
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file,
        language="zh",
    )

print(f"\nSTT 轉錄結果：{transcript.text}")

## Step 2: LLM — 生成回答

### TODO
用 LLM 生成 HP 客服回答。設定一個 system prompt，讓 LLM 扮演 HP 客服人員。

In [ ]:
# =============================================
# TODO: 呼叫 LLM 生成回答（約 3 行）
# 提示：
# 1. 定義 system_prompt，角色是 HP 筆電客服
# 2. 用 llm.invoke() 傳入 system message 和 user message
# 3. 印出回答
#
# from langchain_core.messages import SystemMessage, HumanMessage
# system_prompt = "你是 HP 筆電客服人員，用繁體中文簡潔回答。"
# response = llm.invoke([SystemMessage(content=___), HumanMessage(content=___)])
# print(response.content)
# =============================================

from langchain_core.messages import SystemMessage, HumanMessage

# 你的程式碼寫在這裡
pass

## Step 3: TTS — 文字轉語音

把 LLM 的回答轉成語音播放出來。

In [ ]:
# TTS：將文字回答轉成語音
def text_to_speech(text: str, output_path: str = "response.mp3") -> str:
    """將文字轉成語音檔案"""
    response = client.audio.speech.create(
        model="tts-1",
        voice="nova",
        input=text,
    )
    response.stream_to_file(output_path)
    return output_path

# 測試 TTS
demo_text = "EliteBook 855 搭載 54.5Wh 電池，一般使用約可維持 8 到 10 小時。"
audio_path = text_to_speech(demo_text)
print(f"語音已生成：{audio_path}")
display(Audio(audio_path, autoplay=False))

## Step 4: 用 LangGraph 串成 Pipeline

把 STT → LLM → TTS 三個步驟用 LangGraph 串起來。

In [ ]:
# 定義 Voice Pipeline 的 State
class VoiceState(TypedDict):
    audio_input_path: str    # 輸入音訊路徑
    transcript: str          # STT 轉錄文字
    llm_response: str        # LLM 回答
    audio_output_path: str   # 輸出音訊路徑

In [ ]:
# =============================================
# TODO: 完成三個節點函數（每個約 3 行）
# =============================================

def stt_node(state: VoiceState) -> dict:
    """STT 節點：音訊 → 文字"""
    # =============================================
    # TODO:（約 3 行）
    # 1. 開啟 state["audio_input_path"] 的音訊檔
    # 2. 呼叫 client.audio.transcriptions.create()
    # 3. 回傳 {"transcript": 轉錄結果}
    # =============================================
    pass


def llm_node(state: VoiceState) -> dict:
    """LLM 節點：文字 → 回答"""
    # =============================================
    # TODO:（約 3 行）
    # 1. 定義 system prompt（HP 客服角色）
    # 2. 呼叫 llm.invoke() 傳入 transcript
    # 3. 回傳 {"llm_response": 回答內容}
    # =============================================
    pass


def tts_node(state: VoiceState) -> dict:
    """TTS 節點：文字 → 語音"""
    # =============================================
    # TODO:（約 3 行）
    # 1. 取得 state["llm_response"]
    # 2. 呼叫 text_to_speech() 函數
    # 3. 回傳 {"audio_output_path": 音訊路徑}
    # =============================================
    pass

In [ ]:
# 建立 Graph
graph = StateGraph(VoiceState)

# 加入節點
graph.add_node("stt", stt_node)
graph.add_node("llm", llm_node)
graph.add_node("tts", tts_node)

# 設定起點
graph.set_entry_point("stt")

# =============================================
# TODO: 加入 Edge 串連節點（約 3 行）
# 提示：
# graph.add_edge("stt", ___)
# graph.add_edge("llm", ___)
# graph.add_edge("tts", ___)
# =============================================


# 編譯
voice_app = graph.compile()
print("Voice Pipeline 編譯完成！")

In [ ]:
# 執行 Pipeline
result = voice_app.invoke({
    "audio_input_path": "test_input.mp3",
    "transcript": "",
    "llm_response": "",
    "audio_output_path": "",
})

print("=" * 50)
print("Voice Pipeline 執行結果")
print("=" * 50)
print(f"STT 轉錄：{result['transcript']}")
print(f"LLM 回答：{result['llm_response']}")
print(f"TTS 輸出：{result['audio_output_path']}")
print()

# 播放結果
if result["audio_output_path"]:
    display(Audio(result["audio_output_path"], autoplay=False))

## 延伸挑戰

加入一個 **Normalize 節點**，在 STT 和 LLM 之間，修正常見的 IT 術語拼寫錯誤。

例如：
- "一來不可" → "EliteBook"
- "普若不可" → "ProBook"
- "拍非力恩" → "Pavilion"

提示：
1. 新增一個 `normalize_node` 函數
2. 用簡單的字串替換或 LLM 來修正
3. 修改 Graph 的 Edge：`stt → normalize → llm → tts → END`